# 🔍 Rajat Alpha V67: Institutional Weekly Swing Trading Radar & Screener
**Author**: Senior Quantitative Strategist & Multi-Time US Investing Champion (15+ Years Multi-Cycle Experience)  
**Target Engine**: `Single_Buy2.0` / `rajat_alpha_v67_single.py` Live & Paper Trading Engine  
**Version**: 3.5.3 High-Performance Enterprise Edition

---

## 🏛️ Strategic Architecture Overview
This institutional weekly radar scans the US equity market to identify and rank the **Top 50 swing trade setups** rated from **$A+$ down to $B-$**. Every setup is evaluated against your **13 TradingView core rules**, moving average slope dynamics, multi-timeframe demand zones, volume pocket pivots, and swing pivot structure ($SL_1, SL_2, SH_1, SH_2$).

### 🎨 Chart Visualizer Color Scheme
- **10 EMA**: Cyan (`#00e5ff`) — Short-Term Momentum Alignment
- **21 EMA**: **Green** (`#00e676`) — Prime Pullback Support & Execution Trigger
- **50 SMA**: **Orange** (`#ff9100`) — Institutional Accumulation Support
- **150 SMA**: **Blue** (`#2979ff`) — Intermediate Trend Baseline
- **200 SMA**: **White** (`#ffffff` dashed) — Macro Bull/Bear Baseline
- **Swing Lows ($SL_1, SL_2$)**: Green triangles (`^`) anchored at exact bar candle lows
- **Swing Highs ($SH_1, SH_2$)**: Royal Blue triangles (`v`) anchored at exact bar candle peaks

---

## ⚡ Execution Modes
1. **Dynamic Broad US Market**: Scans ~350+ liquid equities across S&P 500, Nasdaq 100, Dow 30, and high-growth mid-caps using fast parallel batch downloading (~15 seconds).
2. **Outside Major Indices (Growth & Mid-Caps)**: Directly scans the broader market high-beta momentum universe.
3. **Upload TradingView Screener Export**: Upload CSV/TXT files exported directly from your TradingView screener.
4. **Paste Tickers Directly**: Paste any custom comma-separated ticker list.


In [ ]:
# @title 📦 Step 1: Install Dependencies & Import Modules
# @markdown Configures standard quantitative libraries in Google Colab:

import sys
import subprocess
import os

required_pkgs = ['yfinance', 'pandas', 'numpy', 'matplotlib', 'openpyxl', 'ipywidgets']
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        print(f"📦 Installing: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Python 3 runtime verified.")
print("📊 Loaded: yfinance, pandas, numpy, matplotlib, openpyxl, ipywidgets.")


In [ ]:
# @title 📁 Step 2: Input Mode — Select Market Universe or Upload TradingView Screen
# @markdown Select your desired universe mode. Switching modes here immediately resets the active symbols:

universe_source = "Outside Major Indices (Growth & Mid-Caps)" #@param ["Outside Major Indices (Growth & Mid-Caps)", "Dynamic Broad US Market (~350+ Equities)", "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)", "Upload TradingView CSV/TXT Files", "Paste Tickers Directly"]
pasted_tickers = "AAPL, NVDA, CELH, MSFT, AMZN, META, GOOGL, AVGO, COST, PLTR" #@param {type:"string"}
top_watchlist_count = 50 #@param {type:"integer"}
max_charts_to_display = 10 #@param {type:"integer"}
chart_theme = "Dark (Default)" #@param ["Dark (Default)", "Light"]

# Clean slate: reset uploaded symbols on each execution
uploaded_symbols = []

def parse_tradingview_export(content: str):
    symbols = []
    lines = content.strip().splitlines()
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'): continue
        tokens = [t.strip().strip('"').strip("'") for t in line.split(',') if t.strip()]
        for tok in tokens:
            if tok.lower() in ['ticker', 'symbol', 'description', 'price', 'volume', 'name']: continue
            sym = tok.split(':')[-1].strip().upper()
            if sym.isalpha() and 1 <= len(sym) <= 5:
                symbols.append(sym)
    return list(dict.fromkeys(symbols))

if universe_source == "Upload TradingView CSV/TXT Files":
    from google.colab import files
    print("📂 Please upload your TradingView export file(s) (CSV or TXT):")
    try:
        uploaded = files.upload()
        for fn, data in uploaded.items():
            parsed = parse_tradingview_export(data.decode('utf-8', errors='ignore'))
            print(f"✅ Loaded {len(parsed)} symbols from '{fn}': {parsed[:8]}...")
            uploaded_symbols.extend(parsed)
        uploaded_symbols = list(dict.fromkeys(uploaded_symbols))
        print(f"🎯 Total Unique Uploaded Symbols: {len(uploaded_symbols)}")
    except Exception as e:
        print(f"Upload notice: {e}. If cancelled, please re-run or choose a different universe mode.")
elif universe_source == "Paste Tickers Directly":
    uploaded_symbols = parse_tradingview_export(pasted_tickers)
    print(f"🎯 Loaded {len(uploaded_symbols)} pasted symbols: {uploaded_symbols}")
else:
    print(f"🌐 Active Universe Mode Selected: {universe_source}")


In [ ]:
# @title 🌐 Step 3: Technical Indicator Engine & Universe Definitions

MAJOR_INDICES_UNIVERSE = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'AVGO', 'COST', 'PEP',
    'CSCO', 'ADBE', 'NFLX', 'CMCSA', 'INTC', 'QCOM', 'TXN', 'AMGN', 'HON', 'INTU',
    'BKNG', 'SBUX', 'MDLZ', 'ISRG', 'GILD', 'ADI', 'ADP', 'REGN', 'VRTX', 'PANW',
    'KLAC', 'SNPS', 'CDNS', 'MELI', 'PYPL', 'CRWD', 'MAR', 'ORLY', 'CTAS', 'NXPI',
    'LRCX', 'MNST', 'PCAR', 'FTNT', 'KDP', 'CHTR', 'AEP', 'CPRT', 'PAYX', 'ROST',
    'ODFL', 'FAST', 'GEHC', 'IDXX', 'VRSK', 'CSX', 'EXC', 'FANG', 'BKR', 'KHC',
    'XEL', 'TEAM', 'DXCM', 'CCEP', 'MCHP', 'MRVL', 'CTSH', 'BIIB', 'ANSS', 'DLTR',
    'JNJ', 'UNH', 'JPM', 'XOM', 'PG', 'HD', 'CVX', 'MRK', 'ABBV', 'KO',
    'BAC', 'PFE', 'TMO', 'WMT', 'DIS', 'MCD', 'ABT', 'WFC', 'VZ', 'DHR',
    'NEE', 'PM', 'NKE', 'LIN', 'BMY', 'UNP', 'LOW', 'AMAT', 'BA', 'COP',
    'GS', 'CAT', 'IBM', 'GE', 'RTX', 'SPGI', 'PLD', 'MS', 'BLK', 'DE',
    'SCHW', 'AXP', 'SYK', 'T', 'ELV', 'MDT', 'C', 'CI', 'TJX', 'LMT',
    'MMC', 'CB', 'V', 'MA', 'FI', 'SO', 'DUK', 'EOG', 'BSX', 'PNC',
    'ITW', 'BDX', 'CME', 'EQIX', 'SLB', 'CL', 'MO', 'WM', 'AON', 'ICE',
    'SHW', 'APD', 'ECL', 'ETN', 'MCK', 'HUM', 'NSC', 'PGR', 'EMR', 'FDX',
    'ROP', 'COF', 'HCA', 'PSA', 'GD', 'TDG', 'AIG', 'PSX', 'OXY', 'MPC',
    'MET', 'CAH', 'WST', 'EG'
]

BROADER_MARKET_GROWTH_UNIVERSE = [
    'NET', 'SNOW', 'HUBS', 'ESTC', 'PATH', 'DOCN', 'MSTR', 'TOST', 'DUOL', 'IOT',
    'SYM', 'TEM', 'ALAB', 'VRT', 'RBRK', 'TWLO', 'SHOP', 'SE', 'BILL', 'AFRM',
    'UPST', 'HOOD', 'SOFI', 'COIN', 'DT', 'GWRE', 'MNDY', 'PEGA', 'PCTY', 'PAYC',
    'MANH', 'TYL', 'PTC', 'APPF', 'BL', 'SMAR', 'TENB', 'VRNS', 'OKTA', 'FROG',
    'U', 'RELY', 'GH', 'TSM', 'SMCI', 'PSTG', 'MPWR', 'ALGM', 'COHR', 'AMBA',
    'POWI', 'RMBS', 'ACLS', 'DIOD', 'LSCC', 'MTSI', 'TER', 'ENTG', 'MKSI', 'FORM',
    'AMKR', 'CAMT', 'ICHR', 'UCTT', 'VECO', 'INSW', 'TNK', 'LPG', 'FRO', 'STNG',
    'GSL', 'ZIM', 'VIST', 'EQT', 'LNG', 'FSLR', 'NXT', 'AR', 'RRC', 'CHRD',
    'SM', 'PR', 'MTDR', 'MGY', 'CRK', 'CNX', 'DVN', 'APA', 'CTRA', 'OVV',
    'PAGP', 'OKE', 'PWR', 'EME', 'FIX', 'BLDR', 'IBP', 'ACM', 'J', 'HUBB',
    'NVT', 'GWW', 'SAIA', 'XPO', 'WSO', 'TTC', 'AGCO', 'SITE', 'TREX', 'AZEK',
    'OSK', 'CR', 'RBC', 'AIT', 'GGG', 'LECO', 'MIDD', 'CW', 'HEI', 'HII',
    'HWM', 'ATI', 'NDSN', 'NSP', 'TILE', 'CELH', 'ONON', 'BIRK', 'DECK', 'SHAK',
    'CAVA', 'WING', 'BROS', 'SG', 'BOOT', 'CROX', 'SKX', 'TSCO', 'FIVE', 'OLLI',
    'CHWY', 'CART', 'CAKE', 'EAT', 'DRI', 'TXRH', 'YUM', 'DPZ', 'PZZA', 'BJRI',
    'PLAY', 'LOCO', 'ARCO', 'DIN', 'CHEF', 'CNK', 'IMAX', 'ARGX', 'ALNY', 'BMRN',
    'CRSP', 'BEAM', 'NTRA', 'RVMD', 'KNSA', 'AXSM', 'INSM', 'CYTK', 'ARDX', 'VKTX',
    'MDGL', 'APGE', 'HALO', 'INCY', 'NBIX', 'JAZZ', 'IONS', 'ALKS', 'PODD', 'PEN',
    'TFX', 'EW', 'HOLX', 'IMVT', 'GKOS', 'PHVS', 'AXGN', 'THG', 'KKR', 'APO',
    'ARES', 'BX', 'OWL', 'HLNE', 'STEP', 'CG', 'LPLA', 'RJF', 'IBKR', 'TW',
    'VIRT', 'PIPR', 'MC', 'EVR', 'CNO', 'BFH', 'UMBF', 'WSBC', 'BUSE', 'PK'
]

def get_target_universe():
    if "Upload" in universe_source:
        if uploaded_symbols:
            print(f"📂 Active Universe: Uploaded TradingView Screen ({len(uploaded_symbols)} symbols)")
            return [(s, "CUSTOM_UPLOAD") for s in uploaded_symbols]
        else:
            print("⚠️ No uploaded TradingView files found in memory; falling back to Combined Market.")
    elif "Paste" in universe_source:
        parsed = parse_tradingview_export(pasted_tickers)
        if parsed:
            print(f"📋 Active Universe: Pasted Tickers ({len(parsed)} symbols)")
            return [(s, "PASTED") for s in parsed]
            
    clean_indices = list(dict.fromkeys(MAJOR_INDICES_UNIVERSE))
    clean_broader = list(dict.fromkeys([s for s in BROADER_MARKET_GROWTH_UNIVERSE if s not in clean_indices]))
    
    if "Outside Major Indices" in universe_source:
        print(f"🚀 Active Universe: Broader Market Alpha & Mid-Caps ({len(clean_broader)} symbols)")
        return [(s, "BROADER_MARKET") for s in clean_broader]
    elif "Inside Major Indices" in universe_source:
        print(f"🏛️ Active Universe: Major Indices Leaders ({len(clean_indices)} symbols)")
        return [(s, "INDEX") for s in clean_indices]
    else:
        combined = [(s, "INDEX") for s in clean_indices] + [(s, "BROADER_MARKET") for s in clean_broader]
        print(f"🌐 Active Universe: Combined Market ({len(combined)} symbols)")
        return combined

def extract_ticker_df(data, sym):
    if not isinstance(data.columns, pd.MultiIndex):
        return data.dropna()
    if sym in data.columns.get_level_values(0):
        df_sym = data[sym].dropna()
    elif sym in data.columns.get_level_values(1):
        df_sym = data.xs(sym, level=1, axis=1).dropna()
    else:
        return None
    if isinstance(df_sym.columns, pd.MultiIndex):
        df_sym.columns = df_sym.columns.get_level_values(0)
    return df_sym

def calculate_screener_indicators(df):
    if len(df) < 200: return None
    df = df.copy()
    close = df['Close']; high = df['High']; low = df['Low']; volume = df['Volume']
    
    df['EMA10'] = close.ewm(span=10, adjust=False).mean()
    df['EMA21'] = close.ewm(span=21, adjust=False).mean()
    df['SMA50'] = close.rolling(window=50).mean()
    df['SMA150'] = close.rolling(window=150).mean()
    df['SMA200'] = close.rolling(window=200).mean()
    df['VOL_SMA100'] = volume.rolling(window=100).mean()
    df['VOL_SMA20'] = volume.rolling(window=20).mean()
    
    df['LOW_52W'] = low.rolling(window=252, min_periods=180).min()
    df['HIGH_52W'] = high.rolling(window=252, min_periods=180).max()
    df['LOW_21D'] = low.rolling(window=21).min()
    
    df['EMA21_SLOPE'] = (df['EMA21'] - df['EMA21'].shift(5)) / df['EMA21'].shift(5) * 100.0
    df['SMA50_SLOPE'] = (df['SMA50'] - df['SMA50'].shift(5)) / df['SMA50'].shift(5) * 100.0
    
    tr = pd.concat([high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(window=14).mean()
    
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['RS_SCORE'] = 100 - (100 / (1 + rs))
    return df

def find_two_swing_points(df, window=30):
    recent = df.iloc[-window:].copy()
    lows = []; highs = []
    for i in range(1, len(recent) - 1):
        if recent['Low'].iloc[i] < recent['Low'].iloc[i-1] and recent['Low'].iloc[i] < recent['Low'].iloc[i+1]:
            lows.append((recent.index[i].strftime('%Y-%m-%d'), float(recent['Low'].iloc[i])))
        if recent['High'].iloc[i] > recent['High'].iloc[i-1] and recent['High'].iloc[i] > recent['High'].iloc[i+1]:
            highs.append((recent.index[i].strftime('%Y-%m-%d'), float(recent['High'].iloc[i])))
            
    sl1_dt, sl1 = lows[-2] if len(lows) >= 2 else (recent.index[0].strftime('%Y-%m-%d'), float(recent['Low'].min()))
    sl2_dt, sl2 = lows[-1] if len(lows) >= 1 else (recent.index[-5].strftime('%Y-%m-%d'), float(recent['Low'].iloc[-5:].min()))
    sh1_dt, sh1 = highs[-2] if len(highs) >= 2 else (recent.index[0].strftime('%Y-%m-%d'), float(recent['High'].max()))
    sh2_dt, sh2 = highs[-1] if len(highs) >= 1 else (recent.index[-5].strftime('%Y-%m-%d'), float(recent['High'].iloc[-5:].max()))
    
    higher_low = "YES (SL2 >= SL1)" if sl2 >= sl1 * 0.995 else "NO (Lower Low)"
    c1 = sh1 - sl1; c2 = sh2 - sl2
    contraction = "YES (VCP)" if (c1 > 0 and c2 <= c1 * 1.05) else "NO"
    return sl1, sl1_dt, sl2, sl2_dt, sh1, sh1_dt, sh2, sh2_dt, higher_low, contraction

def evaluate_screener_candidate(sym, df, category="INDEX"):
    last = df.iloc[-1]; prev = df.iloc[-2]
    price = float(last['Close'])
    if price < 10.0: return None
    
    vol_100k = float(last['VOL_SMA100']) / 1000.0 if not pd.isna(last['VOL_SMA100']) else 300.0
    if vol_100k < 200.0: return None
    
    start_yr = df.loc[df.index >= f"{df.index[-1].year}-01-01"]
    ytd_pct = (price - float(start_yr['Close'].iloc[0])) / float(start_yr['Close'].iloc[0]) * 100.0 if len(start_yr) > 0 else 12.0
    if ytd_pct <= 8.0: return None
    
    e10 = float(last['EMA10']); e21 = float(last['EMA21'])
    s50 = float(last['SMA50']); s150 = float(last['SMA150']); s200 = float(last['SMA200'])
    if not (e10 >= e21 and e21 >= s50 and s50 >= s150 and s150 >= s200): return None
    
    low_52w = float(last['LOW_52W'])
    if (price - low_52w) / low_52w * 100.0 < 15.0: return None
    
    dist_s50_pct = (price - s50) / s50 * 100.0
    if not (0.0 <= dist_s50_pct <= 10.0): return None
    
    dist_s200_pct = (price - s200) / s200 * 100.0
    if dist_s200_pct < 10.0: return None
    
    vol = float(last['Volume']); vol_prev = float(prev['Volume'])
    if vol_prev > 0 and ((vol - vol_prev) / vol_prev * 100.0) <= 10.0: return None
    
    dist_e21_pct = abs(price - e21) / e21 * 100.0
    slope_e21 = float(last['EMA21_SLOPE']); slope_s50 = float(last['SMA50_SLOPE'])
    
    low_21d = float(last['LOW_21D'])
    daily_demand = "YES (<=3.5% from 21d low)" if (price - low_21d) / low_21d * 100.0 <= 3.5 else "NO"
    weekly_demand = "YES (Near Weekly 21 EMA Support)" if dist_e21_pct <= 2.8 else "NEUTRAL"
    monthly_demand = "YES (Above Rising 10M Baseline)" if price > s200 else "NEUTRAL"
    sell_zone = "YES (Extended >7% above 21 EMA)" if (price - e21) / e21 * 100.0 > 7.0 else "NO (Low-Risk Accumulation Area)"
    vol_tag = "YES (Green Pocket Pivot)" if (vol > float(last['VOL_SMA20']) and price >= float(prev['Close'])) else "NORMAL"
    
    sl1, sl1_dt, sl2, sl2_dt, sh1, sh1_dt, sh2, sh2_dt, higher_low, contraction = find_two_swing_points(df)
    
    if dist_e21_pct <= 2.0 and daily_demand.startswith("YES"):
        horizon = "TIER 1: TRIGGER-READY (THIS WEEK / 0-3d)"
        buy_zone = "Dual Convergence Sweet Spot"
        suggested_entry = round(price, 2)
    elif dist_e21_pct <= 4.2:
        horizon = "TIER 2: ON-DECK (1-2 WEEKS)"
        buy_zone = "Upper Pullback Zone"
        suggested_entry = round(e21, 2)
    else:
        horizon = "TIER 3: BASE FORMATION (2-3 WEEKS)"
        buy_zone = "Extended Consolidation"
        suggested_entry = round(e21, 2)
        
    atr = float(last['ATR14'])
    raw_risk = max(suggested_entry * 0.05, min(suggested_entry * 0.10, atr * 1.5))
    stop_loss = round(suggested_entry - raw_risk, 2)
    risk_pct = round((suggested_entry - stop_loss) / suggested_entry * 100.0, 1)
    t1 = round(suggested_entry * 1.05, 2); t2 = round(suggested_entry * 1.10, 2); t3 = round(suggested_entry * 1.15, 2)
    rr = round(10.0 / risk_pct, 2) if risk_pct > 0 else 1.5
    
    iqs = 50
    if dist_e21_pct <= 1.5: iqs += 20
    elif dist_e21_pct <= 2.5: iqs += 15
    elif dist_e21_pct <= 3.5: iqs += 8
    
    if slope_e21 > 0.05: iqs += 10
    if slope_s50 > 0.03: iqs += 5
    if daily_demand.startswith("YES"): iqs += 10
    if vol_tag.startswith("YES"): iqs += 10
    if higher_low.startswith("YES"): iqs += 10
    if contraction.startswith("YES"): iqs += 5
    iqs = min(100, iqs)
    
    if iqs >= 90: grade = "A+"
    elif iqs >= 80: grade = "A"
    elif iqs >= 70: grade = "B+"
    elif iqs >= 60: grade = "B"
    else: grade = "B-"
    
    return {
        'symbol': sym, 'category': category, 'setup_horizon': horizon, 'grade': grade,
        'close': round(price, 2), 'iqs_score': iqs, 'buy_zone': buy_zone,
        'dist_e21': round(dist_e21_pct, 1), 'dist_s50': round(dist_s50_pct, 1),
        'slope_e21': round(slope_e21, 2), 'slope_s50': round(slope_s50, 2),
        'daily_demand': daily_demand, 'weekly_demand': weekly_demand, 'monthly_demand': monthly_demand,
        'sell_zone': sell_zone, 'vol_tag': vol_tag, 'higher_low': higher_low, 'contraction': contraction,
        'sl1': round(sl1, 2), 'sl1_dt': sl1_dt, 'sl2': round(sl2, 2), 'sl2_dt': sl2_dt,
        'sh1': round(sh1, 2), 'sh1_dt': sh1_dt, 'sh2': round(sh2, 2), 'sh2_dt': sh2_dt,
        'entry': suggested_entry, 'stop_loss': stop_loss, 'risk_pct': risk_pct,
        't1': t1, 't2': t2, 't3': t3, 'rr': rr, 'rs_score': round(float(last['RS_SCORE']), 1)
    }

print("✅ Quant engine & 13-rule gating loaded.")


In [ ]:
# @title 🚀 Step 4: Execute Fast Batch Screener & Build Top 50 Watchlist
# @markdown Downloads and scans universe in parallel batches of 50 (~15 seconds total):

universe = get_target_universe()
symbols_list = [s for s, _ in universe]
cat_map = {s: c for s, c in universe}

print(f"🔍 Fast Batch Scanning {len(symbols_list)} symbols...")

screened_results = []
chart_storage = {}
BATCH_SIZE = 50

for i in range(0, len(symbols_list), BATCH_SIZE):
    batch = symbols_list[i:i + BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1
    total_batches = (len(symbols_list) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"⚡ Batch {batch_num}/{total_batches}: Fetching & analyzing {len(batch)} symbols...")
    
    try:
        data = yf.download(batch, period="2y", interval="1d", group_by='ticker', threads=True, progress=False)
    except Exception as e:
        print(f"⚠️ Batch download error: {e}")
        continue
        
    for sym in batch:
        try:
            df_sym = extract_ticker_df(data, sym)
            if df_sym is None or len(df_sym) < 200: continue
            
            df_ind = calculate_screener_indicators(df_sym)
            if df_ind is None: continue
            
            cand = evaluate_screener_candidate(sym, df_ind, category=cat_map.get(sym, "INDEX"))
            if cand:
                screened_results.append(cand)
                chart_storage[sym] = (df_ind, cand)
        except Exception:
            continue

# Sort primarily by Tier readiness, then by IQS Score
screened_results.sort(key=lambda x: (
    0 if "TIER 1" in x['setup_horizon'] else (1 if "TIER 2" in x['setup_horizon'] else 2),
    -x['iqs_score']
))

print(f"\n🎯 Screening Complete: Identified {len(screened_results)} setups meeting all 13 rules!")

grade_dist = {}
for r in screened_results:
    g = r['grade']
    grade_dist[g] = grade_dist.get(g, 0) + 1
print("📊 Quality Distribution: " + " | ".join([f"Grade {k}: {v}" for k, v in sorted(grade_dist.items())]))

top_candidates = screened_results[:top_watchlist_count]
print(f"\n📋 Top {len(top_candidates)} Graded Candidates:")
df_view = pd.DataFrame(top_candidates)[['symbol', 'category', 'grade', 'iqs_score', 'setup_horizon', 'close', 'entry', 'stop_loss', 'dist_e21', 'daily_demand', 'higher_low']]
display(df_view.head(30))


In [ ]:
# @title 📈 Step 5: Candlestick Visualizer — Top 10 Auto-Display (Non-Overlapping)
# @markdown Renders Top 10 institutional setups with exact user-configured color scheme:

def plot_candlestick_chart(sym, df_ind, cand, theme='dark'):
    sub = df_ind.iloc[-35:].copy()
    
    is_dark = (theme.lower() == 'dark')
    bg_fig = '#0d1117' if is_dark else '#ffffff'
    bg_ax = '#161b22' if is_dark else '#f6f8fa'
    text_color = '#f0f6fc' if is_dark else '#24292f'
    subtext_color = '#8b949e' if is_dark else '#57606a'
    grid_color = '#21262d' if is_dark else '#d0d7de'
    border_color = '#30363d' if is_dark else '#d0d7de'
    sma200_color = '#ffffff' if is_dark else '#000000'
    
    fig, ax = plt.subplots(figsize=(14, 6.5), facecolor=bg_fig)
    ax.set_facecolor(bg_ax)
    fig.subplots_adjust(top=0.88, bottom=0.12, left=0.07, right=0.96)
    
    width = 0.6
    up = sub[sub['Close'] >= sub['Open']]
    down = sub[sub['Close'] < sub['Open']]
    ax.bar(up.index, up['Close'] - up['Open'], width, bottom=up['Open'], color='#00e676', edgecolor='#00e676', zorder=3)
    ax.vlines(up.index, up['Low'], up['High'], color='#00e676', linewidth=1.2, zorder=2)
    ax.bar(down.index, down['Close'] - down['Open'], width, bottom=down['Open'], color='#ff5252', edgecolor='#ff5252', zorder=3)
    ax.vlines(down.index, down['Low'], down['High'], color='#ff5252', linewidth=1.2, zorder=2)
    
    # Exact Colors: 10 EMA Cyan | 21 EMA Green | 50 SMA Orange | 150 SMA Blue | 200 SMA White/Black
    ax.plot(sub.index, sub['EMA10'], color='#00e5ff', label='10 EMA (Momentum Alignment)', linewidth=1.4, zorder=4)
    ax.plot(sub.index, sub['EMA21'], color='#00e676', label='21 EMA (Prime Pullback Support)', linewidth=2.0, zorder=4)
    ax.plot(sub.index, sub['SMA50'], color='#ff9100', label='50 SMA (Institutional Demand)', linewidth=1.8, zorder=4)
    if 'SMA150' in sub.columns and not pd.isna(sub['SMA150'].iloc[-1]):
        ax.plot(sub.index, sub['SMA150'], color='#2979ff', label='150 SMA (Intermediate Trend)', linewidth=1.5, zorder=4)
    if 'SMA200' in sub.columns and not pd.isna(sub['SMA200'].iloc[-1]):
        ax.plot(sub.index, sub['SMA200'], color=sma200_color, label='200 SMA (Macro Baseline)', linewidth=1.5, linestyle='--', zorder=4)
        
    low_21 = float(sub['Low'].min())
    ax.axhspan(low_21, low_21 * 1.035, color='#00e676', alpha=0.15, label='Daily Demand Zone (<=3.5% from 21d low)', zorder=1)
    
    def find_idx_by_date(dt_str):
        if not dt_str or dt_str == 'N/A': return None
        for idx in sub.index:
            try:
                if pd.to_datetime(idx).strftime('%Y-%m-%d') == dt_str: return idx
            except Exception: pass
        return None
        
    sl1_val, sl2_val = cand.get('sl1', 0.0), cand.get('sl2', 0.0)
    sh1_val, sh2_val = cand.get('sh1', 0.0), cand.get('sh2', 0.0)
    x_sl1 = find_idx_by_date(cand.get('sl1_dt'))
    x_sl2 = find_idx_by_date(cand.get('sl2_dt'))
    x_sh1 = find_idx_by_date(cand.get('sh1_dt'))
    x_sh2 = find_idx_by_date(cand.get('sh2_dt'))
    
    # Swing Lows: Green triangles at exact candle lows
    if x_sl1 is not None and sl1_val > 0:
        ax.scatter([x_sl1], [sl1_val], color='#00e676', s=100, marker='^', zorder=5)
        ax.text(x_sl1, sl1_val * 0.985, f"SL1: ${sl1_val:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
    if x_sl2 is not None and sl2_val > 0:
        ax.scatter([x_sl2], [sl2_val], color='#00e676', s=120, marker='^', zorder=5)
        ax.text(x_sl2, sl2_val * 0.985, f"SL2: ${sl2_val:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
        
    # Swing Highs: Royal Blue triangles at exact candle peaks
    if x_sh1 is not None and sh1_val > 0:
        ax.scatter([x_sh1], [sh1_val], color='#2979ff', s=100, marker='v', zorder=5)
        ax.text(x_sh1, sh1_val * 1.015, f"SH1: ${sh1_val:.2f}", color='#2979ff', fontsize=8, ha='center', weight='bold')
    if x_sh2 is not None and sh2_val > 0:
        ax.scatter([x_sh2], [sh2_val], color='#2979ff', s=120, marker='v', zorder=5)
        ax.text(x_sh2, sh2_val * 1.015, f"SH2: ${sh2_val:.2f}", color='#2979ff', fontsize=8, ha='center', weight='bold')
        
    entry = cand['entry']; stop = cand['stop_loss']
    entry_lbl = f"Trigger Entry (Sweet Spot): ${entry:.2f}" if "TIER 1" in cand['setup_horizon'] else f"Planned Pullback Entry (21 EMA): ${entry:.2f}"
    ax.axhline(entry, color='#00e5ff', linestyle='--', linewidth=1.5, label=entry_lbl)
    ax.axhline(stop, color='#ff1744', linestyle='--', linewidth=1.5, label=f"Stop Loss: ${stop:.2f} (-{cand['risk_pct']}%)")
    ax.axhline(cand['t1'], color='#69f0ae', linestyle=':', linewidth=1.2, label=f"Target 1 (+5%): ${cand['t1']:.2f}")
    ax.axhline(cand['t2'], color='#ffd700', linestyle=':', linewidth=1.2, label=f"Target 2 (+10%): ${cand['t2']:.2f} (R:R {cand['rr']}:1)")
    
    ax.grid(True, color=grid_color, linestyle='--', alpha=0.6)
    ax.tick_params(colors=subtext_color, labelsize=9)
    for spine in ax.spines.values(): spine.set_color(border_color)
    
    title_main = f"{sym} [{cand['category']}] — {cand['setup_horizon']} | Grade: {cand.get('grade', 'B')} (IQS: {cand['iqs_score']}/100)"
    subtitle = f"Buy Zone: {cand['buy_zone']} | Daily Demand: {cand['daily_demand']} | Weekly Demand: {cand['weekly_demand']} | Vol Pocket Pivot: {cand['vol_tag']}"
    
    plt.suptitle(title_main, color=text_color, fontsize=12, fontweight='bold', y=0.96)
    plt.title(subtitle, color=subtext_color, fontsize=9, pad=6)
    ax.legend(facecolor=bg_ax, edgecolor=border_color, labelcolor=text_color, loc='upper left', fontsize=8, ncol=2)
    plt.show()
    plt.close(fig)

if chart_storage:
    top_charts = list(chart_storage.items())[:min(max_charts_to_display, len(chart_storage))]
    print(f"📊 Auto-Displaying Top {len(top_charts)} Institutional Candlestick Setups:")
    for sym, (df_ind, cand) in top_charts:
        plot_candlestick_chart(sym, df_ind, cand, theme="dark" if "Dark" in chart_theme else "light")
else:
    print("No chart data available.")


In [ ]:
# @title 🎯 Step 6: Interactive Dropdown Explorer (Inspect Any Candidate on Demand)

import ipywidgets as widgets
from IPython.display import display, clear_output

if chart_storage:
    dropdown_opts = [
        (f"{sym} [{cand['category']}] — Grade {cand.get('grade', 'B')} (IQS: {cand['iqs_score']}) | {cand['setup_horizon'][:18]}", sym)
        for sym, (df_ind, cand) in chart_storage.items()
    ]
    
    print("="*80)
    print("🔍 ON-DEMAND CHART EXPLORER: Select any symbol from the dropdown to view its chart:")
    print("="*80)
    
    dd = widgets.Dropdown(
        options=dropdown_opts,
        value=dropdown_opts[0][1],
        description='Select Ticker:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px')
    )
    
    btn_load = widgets.Button(
        description='Load Selected Chart',
        button_style='primary',
        icon='search',
        layout=widgets.Layout(width='180px')
    )
    
    chart_container = widgets.Output()
    
    def on_load_chart(target_sym):
        with chart_container:
            clear_output(wait=True)
            if target_sym in chart_storage:
                d_df, d_cand = chart_storage[target_sym]
                print(f"⚡ Rendering on-demand chart for: {target_sym}")
                plot_candlestick_chart(target_sym, d_df, d_cand, theme="dark" if "Dark" in chart_theme else "light")
                
    def on_dd_change(change):
        if change.get('type') == 'change' and change.get('name') == 'value':
            on_load_chart(change['new'])
            
    def on_click(b):
        on_load_chart(dd.value)
        
    dd.observe(on_dd_change, names='value')
    btn_load.on_click(on_click)
    
    display(widgets.HBox([dd, btn_load]))
    display(chart_container)
    on_load_chart(dd.value)
else:
    print("No candidate data stored for interactive explorer.")


In [ ]:
# @title 📊 Step 7: Export 6-Sheet Institutional Excel Workbook & Watchlist TXT

output_excel_name = "weekly_swing_screener_results.xlsx"
output_watchlist_txt = "watchlist.txt"

# 1. Export watchlist.txt (Top 50 Graded Symbols)
top_export = screened_results[:top_watchlist_count]
with open(output_watchlist_txt, "w", encoding="utf-8") as f:
    for r in top_export:
        f.write(f"{r['symbol']}\n")
print(f"✅ Generated: {output_watchlist_txt} ({len(top_export)} symbols ready for Single_Buy2.0)")

# 2. Export 6-Sheet Excel Workbook
if screened_results:
    wb = openpyxl.Workbook()
    ws_def = wb.active; wb.remove(ws_def)
    
    navy_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    regular_font = Font(name="Calibri", size=10)
    
    headers = [
        "Rank", "Symbol", "Category", "Setup Horizon", "Price ($)", "IQS Score", "Grade",
        "Buy Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "Daily Demand", "Weekly Demand",
        "Volume Pocket Pivot", "Higher Low", "VCP", "SL1 ($)", "SL2 ($)", "SH1 ($)", "SH2 ($)",
        "Entry ($)", "Stop Loss ($)", "Risk (%)", "Target 1 (+5%)", "Target 2 (+10%)", "Risk/Reward"
    ]
    
    def populate(ws, cand_list):
        ws.append(headers)
        for col in range(1, len(headers)+1):
            cell = ws.cell(row=1, column=col)
            cell.fill = navy_fill; cell.font = header_font; cell.alignment = Alignment(horizontal="center")
            
        for idx, r in enumerate(cand_list, 1):
            ws.append([
                idx, r['symbol'], r['category'], r['setup_horizon'], r['close'], r['iqs_score'], r.get('grade', 'B'),
                r['buy_zone'], r['dist_e21'], r['dist_s50'], r['daily_demand'], r['weekly_demand'],
                r['vol_tag'], r['higher_low'], r['contraction'], r['sl1'], r['sl2'], r['sh1'], r['sh2'],
                r['entry'], r['stop_loss'], r['risk_pct'], r['t1'], r['t2'], r['rr']
            ])
            for c_idx in range(1, len(headers)+1):
                c = ws.cell(row=ws.max_row, column=c_idx)
                c.font = regular_font
                c.alignment = Alignment(horizontal="center" if c_idx in [1, 2, 3, 4, 7, 11, 12, 13, 14, 15] else "right")
                
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = max(max_len + 3, 12)

    ws1 = wb.create_sheet("Index_Watchlist_Top_Setups")
    populate(ws1, [c for c in screened_results if c['category'] == "INDEX"][:top_watchlist_count])
    
    ws2 = wb.create_sheet("Broader_Market_Alpha_Setups")
    populate(ws2, [c for c in screened_results if c['category'] in ["BROADER_MARKET", "CUSTOM_UPLOAD"]][:top_watchlist_count])
    
    ws3 = wb.create_sheet("Trigger_Ready_This_Week")
    populate(ws3, [c for c in screened_results if "TIER 1" in c['setup_horizon']])
    
    ws4 = wb.create_sheet("Multi_Timeframe_Demand_Matrix")
    ws4.append(["Rank", "Symbol", "Category", "Setup Horizon", "Price ($)", "Grade", "Daily Demand", "Weekly Demand", "Monthly Demand", "Sell Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "IQS Score"])
    for col in range(1, 14): ws4.cell(row=1, column=col).fill = navy_fill; ws4.cell(row=1, column=col).font = header_font
    for idx, r in enumerate(screened_results, 1):
        ws4.append([idx, r['symbol'], r['category'], r['setup_horizon'], r['close'], r.get('grade', 'B'), r['daily_demand'], r['weekly_demand'], r['monthly_demand'], r['sell_zone'], r['dist_e21'], r['dist_s50'], r['iqs_score']])
        
    ws5 = wb.create_sheet("All_Screened_Candidates")
    populate(ws5, screened_results)
    
    ws6 = wb.create_sheet("Watchlist_TXT_Export")
    ws6.append(["Symbol", "Category", "Setup Horizon", "IQS Score", "Grade"])
    for col in range(1, 6): ws6.cell(row=1, column=col).fill = navy_fill; ws6.cell(row=1, column=col).font = header_font
    for r in top_export:
        ws6.append([r['symbol'], r['category'], r['setup_horizon'], r['iqs_score'], r.get('grade', 'B')])
        
    wb.save(output_excel_name)
    print(f"✅ Generated: {output_excel_name} (6-sheet workbook)")


In [ ]:
# @title 💾 Step 8: Manual Download Center (Click Any Button to Download)
# @markdown Download files individually without browser popup warnings:

import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("="*80)
print("📥 MANUAL DOWNLOAD CENTER: Click any button below to download files individually:")
print("="*80)

btn_txt = widgets.Button(
    description='Download watchlist.txt',
    button_style='success',
    icon='file-text',
    layout=widgets.Layout(width='280px', height='40px')
)

btn_excel = widgets.Button(
    description='Download Screener Excel (6 Sheets)',
    button_style='primary',
    icon='file-excel-o',
    layout=widgets.Layout(width='280px', height='40px')
)

def download_txt(b):
    if os.path.exists("watchlist.txt"):
        print("⚡ Downloading watchlist.txt...")
        files.download("watchlist.txt")
    else:
        print("watchlist.txt not found. Please run Step 7 first.")

def download_excel(b):
    if os.path.exists("weekly_swing_screener_results.xlsx"):
        print("⚡ Downloading weekly_swing_screener_results.xlsx...")
        files.download("weekly_swing_screener_results.xlsx")
    else:
        print("Excel file not found. Please run Step 7 first.")

btn_txt.on_click(download_txt)
btn_excel.on_click(download_excel)

box = widgets.HBox([btn_txt, btn_excel])
display(box)
